In [ ]:
pip install scikit-learn seaborn

- model.predict(): Obtiene las salidas del modelo (probabilidades) para el conjunto de datos de validación. Se calculan los steps para asegurar que se procesen todas las imágenes.
- np.argmax(): Convierte las probabilidades en la etiqueta de clase predicha (el índice con el valor más alto).
- validation_generator.classes: Obtiene las etiquetas verdaderas (índices) directamente del generador. Es crucial que shuffle=False se haya usado al crear validation_generator.
- Ajuste de Longitud: Se añade una verificación y ajuste por si el número de muestras no es exactamente divisible por el BATCH_SIZE, lo que podría hacer que predict devuelva un número ligeramente diferente de predicciones que el total de muestras.
- confusion_matrix(): Calcula la matriz usando las etiquetas verdaderas y las predichas.
- classification_report(): Genera un informe útil con métricas como precisión, recall y F1-score para cada clase.
- seaborn.heatmap(): Crea una visualización gráfica de la matriz de confusión para facilitar la interpretación. Las filas representan la clase verdadera y las columnas la clase predicha. La diagonal principal muestra las predicciones correctas.

In [ ]:
# --- Evaluación con Matriz de Confusión ---

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np
import math

print("\nEvaluando el modelo en el conjunto de validación...")

# Es importante resetear el generador antes de predecir si no se usa shuffle=False
# validation_generator.reset() # No necesario si shuffle=False

# Obtener predicciones del modelo para todo el conjunto de validación
# Calculamos los pasos para asegurarnos de cubrir todas las imágenes
validation_steps = math.ceil(validation_generator.samples / BATCH_SIZE)
predictions = model.predict(validation_generator, steps=validation_steps, verbose=1)
# predictions contendrá las probabilidades para cada clase, ej: [[0.1, 0.8, 0.05, 0.05], ...]

# Obtener las etiquetas predichas (el índice de la clase con mayor probabilidad)
predicted_classes = np.argmax(predictions, axis=1)

# Obtener las verdaderas etiquetas
true_classes = validation_generator.classes
# Asegúrate de que el número de predicciones coincida con el número de muestras
if len(predicted_classes) != validation_generator.samples:
     print(f"Advertencia: El número de predicciones ({len(predicted_classes)}) no coincide exactamente con el número de muestras de validación ({validation_generator.samples}). Esto puede ocurrir si el número de muestras no es divisible por el tamaño del lote. Se usarán las primeras {len(predicted_classes)} etiquetas verdaderas.")
     # Recorta las etiquetas verdaderas si es necesario para que coincidan con las predicciones obtenidas
     true_classes = true_classes[:len(predicted_classes)]


# Obtener los nombres de las clases del generador
class_labels = list(validation_generator.class_indices.keys())
print("Clases:", class_labels)


# Calcular la Matriz de Confusión
cm = confusion_matrix(true_classes, predicted_classes)

# Calcular el Reporte de Clasificación (precisión, recall, f1-score)
print("\nReporte de Clasificación:")
print(classification_report(true_classes, predicted_classes, target_names=class_labels))

# Visualizar la Matriz de Confusión
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Matriz de Confusión')
plt.ylabel('Clase Verdadera')
plt.xlabel('Clase Predicha')
plt.tight_layout()
plt.show()

